# SGLD vs SGD

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Stochastic gradient Langevin dynamics for mini-batch Bayesian learning.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/sgld/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's data: y = 1.5 x - 0.5 + 0.3 noise, 200 points
N, noise_sd = 200, 0.3
x = torch.randn(N, dtype=torch.float64)
y = 1.5 * x - 0.5 + noise_sd * torch.randn(N, dtype=torch.float64)
X = torch.stack([x, torch.ones_like(x)], 1)            # parameters theta = (w, b)
prior_var = 10.0

# ── Exact Bayesian posterior (Gaussian prior, known noise) for comparison ───
prec = X.T @ X / noise_sd ** 2 + torch.eye(2, dtype=torch.float64) / prior_var
post_cov = torch.linalg.inv(prec)
post_mean = post_cov @ (X.T @ y) / noise_sd ** 2
print("exact posterior mean =", [round(v, 4) for v in post_mean.tolist()],
      " std =", [round(v, 4) for v in post_cov.diagonal().sqrt().tolist()])


def grad_log_post(theta, idx):
    """Unbiased estimate of grad log p(theta | data) from a minibatch."""
    Xb, yb = X[idx], y[idx]
    g_lik = (Xb.T @ (yb - Xb @ theta)) / noise_sd ** 2 * (N / len(idx))
    return g_lik - theta / prior_var


# ── SGLD: theta <- theta + (eps/2) grad log post + N(0, eps) ────────────────
eps, steps, batch = 2e-4, 20000, 20
theta = torch.zeros(2, dtype=torch.float64)
theta_sgd = torch.zeros(2, dtype=torch.float64)
trace = []
for k in range(steps):
    idx = torch.randint(0, N, (batch,))
    g = grad_log_post(theta, idx)
    theta = theta + 0.5 * eps * g + math.sqrt(eps) * torch.randn(2, dtype=torch.float64)
    theta_sgd = theta_sgd + 0.5 * eps * grad_log_post(theta_sgd, idx)     # same steps, no noise
    if k >= 2000:
        trace.append(theta)
trace = torch.stack(trace)
print("SGLD sample mean     =", [round(v, 4) for v in trace.mean(0).tolist()],
      " std =", [round(v, 4) for v in trace.std(0).tolist()])
print("SGD (no noise) ends  =", [round(v, 4) for v in theta_sgd.tolist()], "-> a point, no uncertainty")
# The noise is what turns an optimizer into a sampler of the posterior. SGLD's spread is a
# little wider than the exact one: minibatch gradient noise adds to it (smaller eps shrinks that).

exact posterior mean = [1.4769, -0.5311]  std = [0.0202, 0.0212]


SGLD sample mean     = [1.4773, -0.5314]  std = [0.0298, 0.0305]
SGD (no noise) ends  = [1.4654, -0.5408] -> a point, no uncertainty
